In [24]:
import ast
from pathlib import Path 

import cv2
import dlib
import numpy as np
import pandas as pd
from deepface import DeepFace
from deepface.modules import preprocessing
from deepface.models.facial_recognition.Facenet import load_facenet128d_model
from tqdm import tqdm

from sklearn.decomposition import PCA

from qdrant_client import QdrantClient 
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models

In [12]:
CLIENT = QdrantClient(host='192.168.0.131', port=6333)

In [8]:
df = pd.read_csv('../../data/test_labeled.csv', index_col=0)
df = df[df['label'].notna()]
df = df.reset_index(drop=True)
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,mouth_left_y,confidence,face_num,frame_num,img_width,img_height,filepath,encoding,label,encoding_facenet
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,0.689,1.000,0,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-9.00431126e-02,1.53707832e-01,2.39705741e-02...",Hugh Laurie,"[1.7062506675720215, -1.7491936683654785, -0.3..."
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,0.778,1.000,0,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.11312114,0.1268733,0.00903357,-0.03153913,...",Robert Sean Leonard,"[-0.021657362580299377, -1.5123425722122192, -..."
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,0.409,1.000,0,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.02710779,0.19948913,0.03290576,-0.13188788...",Robert Sean Leonard,"[-0.28194111585617065, -1.3996846675872803, -0..."
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,0.383,0.998,1,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-5.59481643e-02,7.91650563e-02,2.86144577e-02...",Hugh Laurie,"[0.12560777366161346, -2.5007336139678955, 0.0..."
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,0.412,0.999,0,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-4.07901593e-02,1.11157432e-01,7.99753368e-02...",Hugh Laurie,"[0.4325483739376068, -1.5880738496780396, -0.0..."


In [9]:
encodings = []
cap = cv2.VideoCapture(df.at[0, 'filepath'])
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    x1 = int(row['x1'] * row['img_width'])
    y1 = int(row['y1'] * row['img_height'])
    x2 = int(row['x2'] * row['img_width'])
    y2 = int(row['y2'] * row['img_height'])
    face = frame[y1:y2, x1:x2]
    encoding = DeepFace.represent(face, 
                                  model_name='Facenet512', 
                                  detector_backend='skip', 
                                  enforce_detection=False,
                                  normalization='Facenet2018',
                                  max_faces=1,
                                  align=True)
    encodings.append(encoding)


100%|██████████| 344/344 [02:03<00:00,  2.79it/s]


In [10]:
df['encoding_facenet512'] = [x[0]['embedding'] for x in encodings]
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,confidence,face_num,frame_num,img_width,img_height,filepath,encoding,label,encoding_facenet,encoding_facenet512
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,1.000,0,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-9.00431126e-02,1.53707832e-01,2.39705741e-02...",Hugh Laurie,"[1.7062506675720215, -1.7491936683654785, -0.3...","[1.7062504291534424, -1.7491933107376099, -0.3..."
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,1.000,0,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.11312114,0.1268733,0.00903357,-0.03153913,...",Robert Sean Leonard,"[-0.021657362580299377, -1.5123425722122192, -...","[-0.021657384932041168, -1.5123451948165894, -..."
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,1.000,0,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.02710779,0.19948913,0.03290576,-0.13188788...",Robert Sean Leonard,"[-0.28194111585617065, -1.3996846675872803, -0...","[-0.28193968534469604, -1.3996847867965698, -0..."
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,0.998,1,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-5.59481643e-02,7.91650563e-02,2.86144577e-02...",Hugh Laurie,"[0.12560777366161346, -2.5007336139678955, 0.0...","[0.12560713291168213, -2.500732898712158, 0.08..."
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,0.999,0,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-4.07901593e-02,1.11157432e-01,7.99753368e-02...",Hugh Laurie,"[0.4325483739376068, -1.5880738496780396, -0.0...","[0.43254902958869934, -1.5880749225616455, -0...."


In [13]:
df_labeled = df.copy()
df_labeled = df_labeled.assign(predicted_facenet=None, predicted_facenet_score=None)
for idx, row in tqdm(df_labeled.iterrows(), total=df_labeled.shape[0]):
    encoding = row['encoding_facenet512']
    response = CLIENT.query_points(collection_name='Headshots_512',
                                   query=encoding,
                                   limit=1)
    df_labeled.at[idx, 'predicted_facenet'] = response.points[0].payload['name']
    df_labeled.at[idx, 'predicted_facenet_score'] = response.points[0].score

100%|██████████| 344/344 [00:02<00:00, 132.77it/s]


In [14]:
df_labeled.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,frame_num,img_width,img_height,filepath,encoding,label,encoding_facenet,encoding_facenet512,predicted_facenet,predicted_facenet_score
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-9.00431126e-02,1.53707832e-01,2.39705741e-02...",Hugh Laurie,"[1.7062506675720215, -1.7491936683654785, -0.3...","[1.7062504291534424, -1.7491933107376099, -0.3...",Hugh Laurie,0.388661
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.11312114,0.1268733,0.00903357,-0.03153913,...",Robert Sean Leonard,"[-0.021657362580299377, -1.5123425722122192, -...","[-0.021657384932041168, -1.5123451948165894, -...",Robert Sean Leonard,0.629558
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.02710779,0.19948913,0.03290576,-0.13188788...",Robert Sean Leonard,"[-0.28194111585617065, -1.3996846675872803, -0...","[-0.28193968534469604, -1.3996847867965698, -0...",Robert Sean Leonard,0.766441
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-5.59481643e-02,7.91650563e-02,2.86144577e-02...",Hugh Laurie,"[0.12560777366161346, -2.5007336139678955, 0.0...","[0.12560713291168213, -2.500732898712158, 0.08...",Hugh Laurie,0.650903
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-4.07901593e-02,1.11157432e-01,7.99753368e-02...",Hugh Laurie,"[0.4325483739376068, -1.5880738496780396, -0.0...","[0.43254902958869934, -1.5880749225616455, -0....",Hugh Laurie,0.543628


In [15]:
correct = df_labeled[df_labeled['label'] == df_labeled['predicted_facenet']]
incorrect = df_labeled[df_labeled['label'] != df_labeled['predicted_facenet']]

In [17]:
correct.shape[0]/df_labeled.shape[0]

0.9069767441860465

In [18]:
CLIENT.recreate_collection(collection_name='Headshots_128',
                           vectors_config=VectorParams(size=128, distance=Distance.COSINE))

True

In [19]:
encodings = []
cap = cv2.VideoCapture(df.at[0, 'filepath'])
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    x1 = int(row['x1'] * row['img_width'])
    y1 = int(row['y1'] * row['img_height'])
    x2 = int(row['x2'] * row['img_width'])
    y2 = int(row['y2'] * row['img_height'])
    face = frame[y1:y2, x1:x2]
    encoding = DeepFace.represent(face, 
                                  model_name='Facenet', 
                                  detector_backend='skip', 
                                  enforce_detection=False,
                                  normalization='Facenet',
                                  max_faces=1,
                                  align=True)
    encodings.append(encoding)

100%|██████████| 344/344 [02:05<00:00,  2.73it/s]


In [20]:
df['encoding_facenet128'] = [x[0]['embedding'] for x in encodings]
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,face_num,frame_num,img_width,img_height,filepath,encoding,label,encoding_facenet,encoding_facenet512,encoding_facenet128
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,0,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-9.00431126e-02,1.53707832e-01,2.39705741e-02...",Hugh Laurie,"[1.7062506675720215, -1.7491936683654785, -0.3...","[1.7062504291534424, -1.7491933107376099, -0.3...","[-1.775043249130249, -1.0839567184448242, -0.4..."
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,0,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.11312114,0.1268733,0.00903357,-0.03153913,...",Robert Sean Leonard,"[-0.021657362580299377, -1.5123425722122192, -...","[-0.021657384932041168, -1.5123451948165894, -...","[-0.9540096521377563, -0.92826247215271, -0.37..."
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,0,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.02710779,0.19948913,0.03290576,-0.13188788...",Robert Sean Leonard,"[-0.28194111585617065, -1.3996846675872803, -0...","[-0.28193968534469604, -1.3996847867965698, -0...","[-0.7978284358978271, -1.3113147020339966, -0...."
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,1,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-5.59481643e-02,7.91650563e-02,2.86144577e-02...",Hugh Laurie,"[0.12560777366161346, -2.5007336139678955, 0.0...","[0.12560713291168213, -2.500732898712158, 0.08...","[-0.5998601913452148, -1.231856107711792, -0.8..."
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,0,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-4.07901593e-02,1.11157432e-01,7.99753368e-02...",Hugh Laurie,"[0.4325483739376068, -1.5880738496780396, -0.0...","[0.43254902958869934, -1.5880749225616455, -0....","[-0.7478652000427246, -2.8572022914886475, -0...."


In [21]:
df_labeled = df.copy()
df_labeled = df_labeled.assign(predicted_facenet=None, predicted_facenet_score=None)
for idx, row in tqdm(df_labeled.iterrows(), total=df_labeled.shape[0]):
    encoding = row['encoding_facenet128']
    response = CLIENT.query_points(collection_name='Headshots_128',
                                   query=encoding,
                                   limit=1)
    df_labeled.at[idx, 'predicted_facenet'] = response.points[0].payload['name']
    df_labeled.at[idx, 'predicted_facenet_score'] = response.points[0].score

100%|██████████| 344/344 [00:02<00:00, 140.00it/s]


In [22]:
correct = df_labeled[df_labeled['label'] == df_labeled['predicted_facenet']]
correct.shape[0]/df_labeled.shape[0]

0.936046511627907

In [27]:
model = load_facenet128d_model()

In [28]:
encodings = []
cap = cv2.VideoCapture(df.at[0, 'filepath'])
for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, row['frame_num'])
    ret, frame = cap.read()
    x1 = int(row['x1'] * row['img_width'])
    y1 = int(row['y1'] * row['img_height'])
    x2 = int(row['x2'] * row['img_width'])
    y2 = int(row['y2'] * row['img_height'])
    face = frame[y1:y2, x1:x2]
    f = preprocessing.normalize_input(preprocessing.resize_image(face, (160, 160)), normalization='Facenet')
    encoding = model(f)[0]
    encodings.append(encoding)


  0%|          | 0/344 [00:00<?, ?it/s]

100%|██████████| 344/344 [02:04<00:00,  2.77it/s]


In [33]:
df['encoding_facenet_custom'] = [x._numpy() for x in encodings]
df.head()

,x1,y1,x2,y2,right_eye_x,right_eye_y,left_eye_x,left_eye_y,nose_x,nose_y,...,frame_num,img_width,img_height,filepath,encoding,label,encoding_facenet,encoding_facenet512,encoding_facenet128,encoding_facenet_custom
0,0.299,0.066,0.659,0.893,0.401,0.333,0.572,0.356,0.485,0.503,...,4224,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-9.00431126e-02,1.53707832e-01,2.39705741e-02...",Hugh Laurie,"[1.7062506675720215, -1.7491936683654785, -0.3...","[1.7062504291534424, -1.7491933107376099, -0.3...","[-1.775043249130249, -1.0839567184448242, -0.4...","[-2.1931021, -1.1702775, -0.05186031, -0.28516..."
1,0.391,0.129,0.748,0.957,0.571,0.410,0.714,0.471,0.680,0.595,...,5304,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.11312114,0.1268733,0.00903357,-0.03153913,...",Robert Sean Leonard,"[-0.021657362580299377, -1.5123425722122192, -...","[-0.021657384932041168, -1.5123451948165894, -...","[-0.9540096521377563, -0.92826247215271, -0.37...","[-1.0988036, -1.0765367, -0.34898892, 1.293547..."
2,0.181,0.220,0.292,0.486,0.215,0.343,0.265,0.320,0.251,0.386,...,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-0.02710779,0.19948913,0.03290576,-0.13188788...",Robert Sean Leonard,"[-0.28194111585617065, -1.3996846675872803, -0...","[-0.28193968534469604, -1.3996847867965698, -0...","[-0.7978284358978271, -1.3113147020339966, -0....","[-1.6257749, -1.0693301, -0.90805095, 1.518655..."
3,0.529,0.152,0.656,0.454,0.546,0.283,0.595,0.273,0.561,0.341,...,5496,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-5.59481643e-02,7.91650563e-02,2.86144577e-02...",Hugh Laurie,"[0.12560777366161346, -2.5007336139678955, 0.0...","[0.12560713291168213, -2.500732898712158, 0.08...","[-0.5998601913452148, -1.231856107711792, -0.8...","[-1.3823948, -1.3583479, -1.617066, 2.7094266,..."
4,0.647,0.199,0.757,0.491,0.682,0.315,0.734,0.301,0.718,0.365,...,5664,1920,1080,/home/amos/media/tv/House.MD.2004.S01-08.1080p...,"[-4.07901593e-02,1.11157432e-01,7.99753368e-02...",Hugh Laurie,"[0.4325483739376068, -1.5880738496780396, -0.0...","[0.43254902958869934, -1.5880749225616455, -0....","[-0.7478652000427246, -2.8572022914886475, -0....","[-1.1185799, -2.895713, -0.67029697, 2.873408,..."


In [34]:
df_labeled = df.copy()
df_labeled = df_labeled.assign(predicted_facenet=None, predicted_facenet_score=None)
for idx, row in tqdm(df_labeled.iterrows(), total=df_labeled.shape[0]):
    encoding = row['encoding_facenet_custom']
    response = CLIENT.query_points(collection_name='Headshots_128',
                                   query=encoding,
                                   limit=1)
    df_labeled.at[idx, 'predicted_facenet'] = response.points[0].payload['name']
    df_labeled.at[idx, 'predicted_facenet_score'] = response.points[0].score

100%|██████████| 344/344 [00:02<00:00, 141.11it/s]


In [35]:
correct = df_labeled[df_labeled['label'] == df_labeled['predicted_facenet']]
correct.shape[0]/df_labeled.shape[0]

0.9447674418604651